# 02 RAG 问答链路

**目标**：检索 → 拼 prompt → 生成。

RAG 的价值不在「用了大模型」，而在**答案有出处、可追溯**。
所以 prompt 里必须强制模型只依据资料回答，并列出引用。

In [1]:
import sys, os
sys.path.insert(0, '../..')
from src.rag_core import KnowledgeBase, build_prompt, call_llm

kb = KnowledgeBase('../../data/kb').build()
print(kb.stats())

{'documents': 3, 'chunks': 16, 'dim': 5071, 'reduced': False}


## 1. 检索

In [2]:
question = '矩阵维度不匹配怎么排查'
hits = kb.search(question, k=3)

for i, h in enumerate(hits, 1):
    print(f"[{i}] {h['score']:.3f}  {h['source']} · {h['title']}")


[1] 0.195  PyTorch常见错误.md · 张量形状不匹配
[2] 0.019  模型部署指南.md · 服务的可观测性
[3] 0.000  PyTorch常见错误.md · 梯度爆炸与梯度消失


## 2. 拼装 prompt

注意三条约束：只依据资料、不知道就说不知道、必须列出处。
**不加这三条，模型就会开始编。**

In [3]:
prompt = build_prompt(question, hits)
print(prompt[:1200])

你是 AI 实训平台的助教。请严格依据用户给出的资料回答问题：
1. 只使用资料中的信息，资料里没有的内容不要补充，不要编造；
2. 资料不足时也要先回答资料里已有的部分，不要只回一句结论；
3. 先分条写排查步骤或做法，最后单独一行用「依据：」列出引用的来源文件名。

[资料1] 来源：PyTorch常见错误.md · 张量形状不匹配
## 张量形状不匹配

最常见的报错是矩阵乘法维度不匹配，典型信息是 `mat1 and mat2 shapes cannot be multiplied`。

以 `x @ y` 为例，要求 x 的最后一维等于 y 的倒数第二维。如果 x 是形状 (3, 4)，y 是 (3, 5)，那么 x 的最后一维是 4，y 的倒数第二维是 3，两者不相等，无法相乘。正确的做法是把 y 转置成 (4, 5)，或者检查上游网络层的输出维度。

在卷积网络中，全连接层之前必须把特征图展平。如果卷积层输出是 (batch, 64, 7, 7)，进入全连接层前要用 `x.view(x.size(0), -1)` 变成 (batch, 3136)。忘记展平就会出现维度不匹配。

[资料2] 来源：模型部署指南.md · 服务的可观测性
## 服务的可观测性

上线后至少需要监控三类指标。第一类是业务指标，比如每天的请求量、预测结果的分布，如果分布突然偏移说明数据可能变了。第二类是性能指标，包括 P50 和 P99 延迟、QPS、错误率。第三类是资源指标，GPU 利用率和显存占用。

日志要记录每次请求的输入摘要和输出，方便事后排查。但要注意脱敏，不要把用户隐私数据写进日志。

[资料3] 来源：PyTorch常见错误.md · 梯度爆炸与梯度消失
## 梯度爆炸与梯度消失

训练深层网络时，如果损失突然变成 NaN，通常是梯度爆炸。可以先打印梯度范数确认：遍历 `model.parameters()`，计算每个参数梯度的 norm。

应对手段有三种。梯度裁剪是最直接的，用 `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` 把梯度范数限制在阈值内。降低学习率也有效。改用残差连接或 BatchNorm 能从结构上缓解。

梯度消失的表现相反：损失下降极慢，浅层参数几乎不更

## 3. 调用大模型

模型部署在本地：容器里的 Ollama，通过 **OpenAI 兼容接口**暴露，
所以调用代码和云端完全一样，改的只有 `base_url` 和 `model`。

本地服务不校验 Key，`call_llm` 里传任意非空值即可。
换成云端（DeepSeek / 通义）也只需要改这几个变量，其余代码一行不用动。


In [4]:
import time

# 模型跑在容器内的 Ollama 上，和 8502 页面用的是同一套配置
# 它提供 OpenAI 兼容接口，本地不校验 Key，传个非空值就行
BASE_URL = 'http://ollama:11434/v1'
MODEL = 'qwen2.5-coder:1.5b'
API_KEY = 'ollama'


def ask(prompt, retries=2):
    """结果过短（只回了出处那一行）时重试，保证拿到完整回答。"""
    out = ''
    for _ in range(retries + 1):
        out = call_llm(prompt, API_KEY, base_url=BASE_URL, model=MODEL)
        if len(out.strip()) >= 40:
            break
    return out


t0 = time.time()
answer = ask(prompt)
print(f'（耗时 {time.time() - t0:.1f} 秒）')
print()
print(answer)


（耗时 7.7 秒）

排查矩阵维度不匹配的步骤如下：

1. **检查输入和输出的维度**：
   - 确认 `x` 的最后一维是否等于 `y` 的倒数第二维。
   - 如果不匹配，检查 `x` 和 `y` 的形状是否正确。

2. **调整 `y` 的形状**：
   - 如果 `x` 的最后一维是 4，`y` 的倒数第二维是 3，可以尝试将 `y` 转置成 (4, 5)。
   - 如果 `x` 的最后一维是 4，`y` 的倒数第二维是 3，可以尝试检查上游网络层的输出维度，确保输出维度是 (batch, 3136)。

3. **检查数据预处理**：
   - 确保数据预处理步骤正确，没有错误导致数据形状不匹配。

4. **使用 `torch.nn.utils.clip_grad_norm_`**：
   - 如果梯度范数过大，可以使用 `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` 来限制梯度范数。

5. **调整学习率**：
   - 如果梯度消失，可以尝试降低学习率。

6. **改用残差连接或 BatchNorm**：
   - 如果问题仍然存在，可以尝试改用残差连接或 BatchNorm 来缓解问题。

依据：PyTorch常见错误.md


## 4. 对比：不给资料的模型会怎么答

同一句话直接问，模型只能凭训练记忆回答，给出的是泛泛的通识说法；
RAG 之后，答案里会带上文档里的具体案例和出处。**这是可验证性的差别。**


In [5]:
t0 = time.time()
raw = call_llm(question, API_KEY, base_url=BASE_URL, model=MODEL)
print(f'=== 无资料直接问（{time.time() - t0:.1f} 秒）===')
print(raw[:400])
print()
print('=== 给了资料的 RAG ===')
print(answer[:400])


=== 无资料直接问（11.0 秒）===
排查矩阵维度不匹配的问题通常涉及以下几个步骤：

1. **检查矩阵的维度**：
   - 确认输入的矩阵是否具有正确的维度。例如，如果一个矩阵是 \( m \times n \)，那么它应该有 \( m \) 行和 \( n \) 列。

2. **检查矩阵的大小**：
   - 确认输入的矩阵的大小是否与预期的大小匹配。例如，如果预期的矩阵是 \( 3 \times 3 \)，那么输入的矩阵应该有 3 行和 3 列。

3. **检查矩阵的元素**：
   - 确认输入的矩阵的元素是否与预期的元素匹配。例如，如果预期的矩阵的元素是 1, 2, 3, 4, 5, 6, 7, 8, 9，那么输入的矩阵应该有这些元素。

4. **检查矩阵的索引**：
   - 确认输入的矩阵的索引是否与预期的索引匹配。例如，如果预期的矩阵的元素是 \( A_{i,j} \)，那么输入的矩阵应该有 \( i

=== 给了资料的 RAG ===
排查矩阵维度不匹配的步骤如下：

1. **检查输入和输出的维度**：
   - 确认 `x` 的最后一维是否等于 `y` 的倒数第二维。
   - 如果不匹配，检查 `x` 和 `y` 的形状是否正确。

2. **调整 `y` 的形状**：
   - 如果 `x` 的最后一维是 4，`y` 的倒数第二维是 3，可以尝试将 `y` 转置成 (4, 5)。
   - 如果 `x` 的最后一维是 4，`y` 的倒数第二维是 3，可以尝试检查上游网络层的输出维度，确保输出维度是 (batch, 3136)。

3. **检查数据预处理**：
   - 确保数据预处理步骤正确，没有错误导致数据形状不匹配。

4. **使用 `torch.nn.utils.clip_grad_norm_`**：
   - 如果梯度范数过大，可以使用 `torch.nn.utils.clip_grad_norm_(


## 小结

- RAG = 检索 + 拼装 + 生成，前两步做得不好，换再强的模型也救不回来
- 强制引用出处，是 RAG 相对于纯对话最实际的工程价值
- 下一步：把这个链路包成页面